# McCartneyWHR

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.McCartneyWHR)

class McCartneyWHR(LinearReferenceClock):
    pass



In [3]:
model = pya.models.McCartneyWHR()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'mccartneywhr'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2018
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "McCartney, Daniel L., et al. \"Epigenetic prediction of complex traits and death.\" Genome biology 19.1 (2018): 136."
model.metadata["doi"] = "https://doi.org/10.1186/s13059-018-1514-1"
model.metadata["research_only"] = None
model.metadata["notes"] = None

## Download clock dependencies

In [5]:
supplementary_url = "https://static-content.springer.com/esm/art%3A10.1186%2Fs13059-018-1514-1/MediaObjects/13059_2018_1514_MOESM1_ESM.xlsx"
supplementary_file_name = "mccartney_predictors.xlsx"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
# Additional file 1, Table S9 - Waist-to-Hip ratio (McCartney et al. 2018)
coef_df = pd.read_excel('mccartney_predictors.xlsx', sheet_name='Table S9 - Waist-to-Hip ratio')
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
# The penalised (LASSO) predictor has no intercept term
weights = torch.tensor(coef_df['Beta'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([0.0]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'McCartney, Daniel L., et al. "Epigenetic prediction of complex '
             'traits and death." Genome biology 19.1 (2018): 136.',
 'clock_name': 'mccartneywhr',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13059-018-1514-1',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2018}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg03005261', 'cg21637050', 'cg16728516', 'cg14267725', 'cg06500161', 'cg04804648', 'cg18909879', 'cg25921813', 'cg21020089', 'cg16284674', 'cg15092239', 'cg07769588', 'cg07215298', 'cg09249494', 'cg10474597', 'cg11832534', 'cg25110523', 'cg04453364', 'cg19519737', 'cg18515624', 'cg18054578', 'cg18011760', 'cg08563994', 'cg01616956', 

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ 0.9362],
        [ 1.4919],
        [ 0.3194],
        [-0.1748],
        [ 0.6122],
        [ 0.8324],
        [-0.0945],
        [-0.4978],
        [ 1.0881],
        [-0.8827]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: mccartney_predictors.xlsx
